In [4]:
import glob
import os
import shutil
import zipfile

import numpy as np
import pandas as pd
import xarray as xr

# Preprocessing
In this step we will clean up the data collected in the previous notebook and transform it where needed.

## ENTSO-E

In [66]:
INPUT_DIR = 'data/raw/entsoe'
OUTPUT_PATH = 'data/processed/entsoe_energy.parquet'

BIDDING_ZONES = ['DE_LU', 'DK1']#, 'DK2']

def load_zone(zone: str) -> pd.DataFrame:
    price_pattern = os.path.join(INPUT_DIR, f'{zone}_Price_*.parquet')
    price_parquets = sorted(glob.glob(price_pattern))
    price_df = pd.concat([pd.read_parquet(f) for f in price_parquets])
    load_pattern = os.path.join(INPUT_DIR, f'{zone}_Load_*.parquet')
    load_parquets = sorted(glob.glob(load_pattern))
    load_df = pd.concat([pd.read_parquet(f) for f in load_parquets])
    generation_pattern = os.path.join(INPUT_DIR, f'{zone}_Generation_*.parquet')
    generation_parquets = sorted(glob.glob(generation_pattern))
    generation_df = pd.concat([pd.read_parquet(f) for f in generation_parquets])

    price_df = price_df.asfreq('15 min', method='pad') # upscale to 15 min, the latter data is already at this frequency.
    load_df = load_df.asfreq('15 min', method='pad')
    generation_df = generation_df.fillna(0)             # fill empty data with zeroes.
    generation_df.columns = [' - '.join(col).strip() for col in generation_df.columns] # flatten multindex

    df = pd.concat([price_df, load_df, generation_df], axis=1, join='inner')
    return df


frames = [load_zone(z) for z in BIDDING_ZONES]
df = pd.concat(frames, axis=1, keys=BIDDING_ZONES)
df.to_parquet(OUTPUT_PATH)

## Copernicus
For the weather data we have a bit more to do.
  - convert the netcdf files (or archives of netcdf files) to pandas dataframes
  - We have to combine the monthly data for each variable into one dataframe per country, like we already have for the energy data.
  - We have the wind speed given as u and v components, which we need to transform to one single wind speed
  - We have one data point per 0.25° by 0.25° sector for each of the variables which we will take a mean of.
  - The solar radiation is given as the prefix sum of the sequence, which we will differentiate back into the sequence.

In [ ]:

INPUT_DIR = 'data/raw/era5'
OUTPUT_PATH = 'data/processed/era5_weather.parquet'

COUNTRIES = ['germany', 'luxembourg', 'denmark']


def ensure_unzipped(path: str) -> list[str]:
    if not zipfile.is_zipfile(path):
        return [path]
 
    extract_dir = path + '_unzipped'
    if not os.path.isdir(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path) as zf:
            zf.extractall(extract_dir)

    nc_files = sorted(glob.glob(os.path.join(extract_dir, '*.nc')))
    if not nc_files:
        raise ValueError(f'ZIP {path} does not contain .nc-files')
    return nc_files


def stream_group(nc_path: str) -> str:
    '''group data of the same type across months'''
    name = os.path.basename(nc_path).lower()
    if 'accum' in name:
        return 'accum'
    if 'instant' in name:
        return 'instant'
    return 'default'


def load_country(country: str) -> xr.Dataset:
    pattern = os.path.join(INPUT_DIR, f'era5_{country}_*.nc')
    zip_or_nc_files = sorted(glob.glob(pattern))
    if not zip_or_nc_files:
        raise FileNotFoundError(f'No file found for pattern: {pattern}')
 
    all_nc_files: list[str] = []
    for f in zip_or_nc_files:
        all_nc_files.extend(ensure_unzipped(f))
 
    groups: dict[str, list[str]] = {}
    for nc_path in all_nc_files:
        groups.setdefault(stream_group(nc_path), []).append(nc_path)
 
    group_datasets = [
        xr.open_mfdataset(sorted(paths), combine='by_coords')
        for paths in groups.values()
    ]
    ds = xr.merge(group_datasets, compat='override', join='inner')
 
    if 'valid_time' in ds.coords and 'time' not in ds.coords:
        ds = ds.rename({'valid_time': 'time'})
 
    return ds


def area_mean(ds: xr.Dataset) -> xr.Dataset:
    return ds.mean(dim=['latitude', 'longitude'], skipna=True) # Although the more southern latitudes will have slightly larger sectors,
                                                               # we just take the mean directly. The countries are span a bit more than 10°
                                                               # so this simplification shouldn't be a big problem.
                                                               # We also consider values for sectors, which aren't even part of the countries area,
                                                               # as we take the mean across the whole bounding box.
                                                               # This isn't clean but shouldn't be a huge issue.


def deaccumulate_ssrd(df: pd.DataFrame) -> pd.Series:
    s = df['ssrd'].copy()
    day = df.index.floor('D')
    hourly_diff = s.groupby(day).diff()
    # special case for first hour of the day
    is_first_hour = df.index.hour == 0
    hourly_diff[is_first_hour] = s[is_first_hour]
    return hourly_diff.clip(lower=0)

def cleanup_unzipped() -> None:
    pattern = os.path.join(INPUT_DIR, '*_unzipped')
    for d in glob.glob(pattern):
        shutil.rmtree(d)

def process_country(country: str) -> None: #pd.DataFrame:
    ds = load_country(country)
    
    ds = ds.copy()
    ds['wind_speed_10m'] = np.sqrt(ds['u10'] ** 2 + ds['v10'] ** 2)    # As this is a non-linear operation we have to perform it before taking
    ds['wind_speed_100m'] = np.sqrt(ds['u100'] ** 2 + ds['v100'] ** 2) # the mean otherwise opposing winds within a country would cancel out.

    ds = area_mean(ds)

    df = ds.to_dataframe()

    df['temperature_2m_celsius'] = df["t2m"] - 273.15  # Kelvin -> Celsius

    df = df.sort_index()
    df['solar_radiation_Jm2'] = deaccumulate_ssrd(df)

    df['country'] = country
    cleanup_unzipped()
    return df[[
        'country', 'wind_speed_10m', 'wind_speed_100m',
        'temperature_2m_celsius', 'solar_radiation_Jm2',
    ]]


os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

frames = [process_country(c) for c in COUNTRIES]
combined = pd.concat(frames)
combined.index.name = "time"

combined.to_parquet(OUTPUT_PATH)
print(f"saved: {OUTPUT_PATH}  ({len(combined)} lines)")
print(combined.head())


saved: data/processed/era5_weather.parquet  (75984 lines)
                     country  wind_speed_10m  wind_speed_100m  \
time                                                            
2019-01-01 00:00:00  germany        4.725358         7.678353   
2019-01-01 01:00:00  germany        4.927474         7.979616   
2019-01-01 02:00:00  germany        5.246872         8.428451   
2019-01-01 03:00:00  germany        5.545803         8.862441   
2019-01-01 04:00:00  germany        5.813067         9.279086   

                     temperature_2m_celsius  solar_radiation_Jm2  
time                                                              
2019-01-01 00:00:00                5.934998                  0.0  
2019-01-01 01:00:00                5.899261                  0.0  
2019-01-01 02:00:00                5.887115                  0.0  
2019-01-01 03:00:00                5.850494                  0.0  
2019-01-01 04:00:00                5.842163                  0.0  
